In [1]:
import sys
print(sys.version)

# The collab version of python is incompatible with vllm 0.5.3.
# Therefore, I will be using whatever pip gives me that is compatible
# If you need us to work with a specific version, you may need to specify the
# version details of the dependencies as well. Or if you are working with conda
# just send us the env


3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]


In [2]:
!pip install vllm
# Removed this:
# ==0.5.3.post1


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 5.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 370.3/370.3 MB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 355.0/355.0 kB 31.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.0/183.0 kB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 130.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 62.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 93.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 899.7/899.7 MB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2

In [ ]:
import argparse
import json
from pathlib import Path

import torch
from vllm import LLM, SamplingParams

DEVICE_NUM = torch.cuda.device_count()

# --------------------------------------------------------------------------------------------------------------#
# Prompt templates
# --------------------------------------------------------------------------------------------------------------#
SYSPROMPT = "You are a helpful medical assistant, expert clinical argumentation and who knows Spanish.\n"
PROMPT = """
You are given a justification for a multiple-choice medical exam question. The justification is written in an option-oriented style, where the reasoning is structured around eliminating or supporting specific answer choices. Your task is to rewrite this justification in a natural, clinical reasoning format, as a clinician would explain their diagnostic thinking in real-life practice.

Instructions:
- Do not refer to options or answer numbers (e.g., avoid "Option 1 is incorrect").
- Present the reasoning as a flowing, cohesive diagnostic argument.
- Keep all relevant medical content and logic, but phrase it in a neutral, clinician-oriented way.
- Use appropriate clinical language, as would be found in case presentations, consults, or differential diagnosis discussions.
- Avoid exam-like phrasing such as “this makes option X incorrect.”

Some evidences are marked in the argumentation between '#' or '@' characters:
- '@' indicates an attack relation.
- '#' may indicate a support relation or an attack relation if indicated in parenthesis (i.e.: #evidence#(rechaza)#).

Example:

Options:
1. - Carcinoma indiferenciado nasofaríngeo.
2. - Adenocarcinoma de base de lengua.
3. - Carcinoma mucoepidermoide de hipofaringe.
4. - Carcinoma epidermoide de laringe.

Argumentation:
correcta 4: ante un paciente de edad media, varón, fumador, que presenta disfonía y una masa laterocervical asociada a disnea, debemos pensar en carcinoma epidermoide de laringe como primera sospecha diagnóstica. #fumador# #disfonía# #masa laterocervical pétrea y cierta dificultad respiratoria#      incorrecta 1: el carcinoma de nasofaringe no tendría que cursar con disfonía por su localización @disfonía@ y tampoco se ha relacionado con el tabaquismo @fumador@      incorrectas 2 y 3: estas localizaciones anatómicas tampoco deberían cursar con disfonía@disfonía@

Output Example:
En un paciente de mediana edad, varón, fumador, que consulta por disfonía y presenta una masa laterocervical pétrea acompañada de cierta dificultad respiratoria, la sospecha diagnóstica principal debe orientarse hacia un carcinoma epidermoide de laringe. La presencia de disfonía sugiere afectación directa de las cuerdas vocales o estructuras adyacentes de la laringe, lo cual no es habitual en neoplasias de localización nasofaríngea, lingual posterior o hipofaríngea. Además, el antecedente de tabaquismo refuerza aún más la posibilidad de una neoplasia laríngea, ya que es un factor de riesgo conocido para este tipo de tumor.


Actual Instance:

Options:
{option_text}

Argumentation:
{argumentation}

Output:
"""


# --------------------------------------------------------------------------------------------------------------#
# --------------------------------------------------------------------------------------------------------------#

def prepare_prompt(question: dict):
    """
    Generate the prompt for a given question.
    :param question: The question dictionary that contains the argumentation and options.
    :return: The formatted prompt as a list of messages.
    """

    argument = question["first_argument"]
    option_text = '\n'.join(f'{idx}. - {option}' for idx, option in enumerate(question["options"], start=1))
    return [
        {"role": "system", "content": SYSPROMPT},
        {"role": "user", "content": PROMPT.format(argumentation=argument, option_text=option_text)}
    ]


def separate_thinking_from_response(generated_text, think_start_token, think_end_token):
    """
    Separate the thinking process from the final response if there is any thinking process.

    :param generated_text: Output generated by the model.
    :param think_start_token: Token indicating the start of the thinking process.
    :param think_end_token: Token indicating the end of the thinking process.
    :return: A tuple (thinking, response). Thinking tokens are removed. If no thinking process, thinking is None. If thinking process is not finished, response is None.
    """
    thinking = None
    response = generated_text

    # If there is a thinking process, separate it
    if generated_text.startswith(think_start_token):
        generated_text = generated_text[len(think_start_token):]

        # Check if thinking process has ended
        if (think_end_token_offset := generated_text.find(think_end_token)) != -1:
            thinking = generated_text[:think_end_token_offset]
            response = generated_text[think_end_token_offset + len(think_end_token):]
        else:
            thinking = generated_text
            response = None

    return thinking, response


def parse_argumentation(input_file: Path, output_folder: Path, model: str, think_start_token: str, think_end_token: str):
    output_folder.mkdir(parents=True, exist_ok=True)

    # Load the llm
    llm = LLM(model=model, tensor_parallel_size=DEVICE_NUM, max_model_len=10000)
    sampling_params = SamplingParams(
        temperature=0.4,
        top_k=10,
        top_p=0.8,
        max_tokens=5000,
    )

    # Process the dataset
    data = json.loads(input_file.read_text(encoding='utf-8'))
    prompts = [prepare_prompt(question) for question in data]

    # Make inference
    outputs = llm.chat(
        prompts,
        sampling_params=sampling_params,
        add_generation_prompt=True,
    )

    # Process outputs
    for question, output in zip(data, outputs):
        generated_text = output.outputs[0].text
        thinking, response = separate_thinking_from_response(generated_text, think_start_token, think_end_token)
        question["neutralized_argumentation"] = response
        question["neutralization_thinking_process"] = thinking

    # Save the results
    (output_folder / f"{model.split('/')[-1]}--{input_file.name}").write_text(json.dumps(data, indent=2, ensure_ascii=False), encoding='utf-8')


if __name__ == '__main__':
    parser = argparse.ArgumentParser()
    parser.add_argument('--input_file', type=Path, default=Path('../../raw/cleaned'))
    parser.add_argument('--output_folder', type=Path, default=Path('../../processed/neutralized'))
    parser.add_argument('--model', type=str, default='google/gemma-3-4b-it')
    parser.add_argument('--think_start_token', type=str, default='<unused94>')
    parser.add_argument('--think_end_token', type=str, default='<unused95>')
    cli_args, _ = parser.parse_known_args()

    parse_argumentation(**vars(cli_args))


INFO 11-18 11:10:55 [__init__.py:216] Automatically detected platform cuda.
INFO 11-18 11:11:13 [utils.py:233] non-default args: {'max_model_len': 10000, 'disable_log_stats': True, 'model': 'google/gemma-3-4b-it'}


config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

INFO 11-18 11:11:35 [model.py:547] Resolved architecture: Gemma3ForConditionalGeneration


`torch_dtype` is deprecated! Use `dtype` instead!


...

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

WARNING 11-18 11:11:46 [__init__.py:3036] We must use the `spawn` multiprocessing start method. Overriding VLLM_WORKER_MULTIPROC_METHOD to 'spawn'. See https://docs.vllm.ai/en/latest/usage/troubleshooting.html#python-multiprocessing for more information. Reasons: CUDA is initialized


RuntimeError: Engine core initialization failed. See root cause above. Failed core proc(s): {}

In [ ]:
from huggingface_hub import notebook_login

notebook_login()
